# MSR-VTT evaluation

Public adaptation of the historical Kaggle workflow. Python modules are byte-identical to captured sources; this notebook replaces embedded source/data payloads with verified files. It has no execution outputs and has not been rerun as a GPU job. Read docs/protocols/kaggle_research_sources.md before running. Attach authorized inputs separately; no test-based model selection.

In [ ]:
# Cell 1 — Dùng thư viện có sẵn; không tải hoặc cài lại môi trường.
import importlib.util
import json
import os
import sys
from pathlib import Path

base = Path(os.environ.get("MSRVTT_WORKSPACE", "/kaggle/working/msrvtt_b3_evaluation"))
CODE_DIR = base / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for _module in ("torch", "numpy", "cv2", "PIL"):
    if importlib.util.find_spec(_module) is None:
        raise RuntimeError(f"Thiếu {_module}: chọn image Python/GPU chuẩn của Kaggle.")
torch = importlib.import_module("torch")
print("Python", sys.version.split()[0], "torch", torch.__version__,
      "CUDA", torch.cuda.is_available())


In [ ]:
# Cấu hình lấy từ lượt chạy đã lưu; các đường dẫn dữ liệu được để trống.
CONFIG = {'INPUT_ROOT': '/kaggle/input',
 'WORK_DIR': '',
 'TEST_CSV': '',
 'TEST_CSV_SHA256': 'e61144b312c8a2fc2126f96504c981f8c23c05b3ba9655a56e2af50ac8a4b16a',
 'CHECKPOINT_SHA256': '3d7e2ba508986c7347380767233b017d4ac4c9fabc135f3457de8625dc833dca',
 'TEST_CACHE_PATH': '',
 'TEST_CACHE_SHA256': 'c721c99d5d29e47661f2cbdd74ff59f1454b265d035a6a11d7faf88444c30095',
 'TEST_CACHE_BYTES': 27131378,
 'TEST_FEATURE_MANIFEST_PATH': '',
 'TEST_FEATURE_MANIFEST_SHA256': '928925476b4bc4b077ff36519d4b6eb5c2fe8f5ba4447a8091c0d10639cbf0ac',
 'TEST_CACHE_PROVENANCE_PATH': '',
 'TEST_CACHE_PROVENANCE_SHA256': '67973ec246b6265b3d927480c5bb90b80559a451851c17979784b804c5e207ea',
 'TEST_CACHE_FINGERPRINT': '192caf0144845f10b52980fd53546b0c7fc4265ac1b249f3b472ad91a6a9c47c',
 'TEST_NUMERIC_FINGERPRINT': '7ade86bebc0c965e7f102db0d74663da344614404c0a4aba07df9b52e52911f2',
 'TEST_CACHE_VALID_FRAMES': 11727,
 'CACHE_SOURCE_ARCHIVE_SHA256': '44ff9576c45ed5b0e8e92b04271da25764ba4b0b2f999496067d64bfd4bbd890',
 'HEAD_PATH': '',
 'HEAD_SHA256': '374a56f283203051a8c27285c4d5b37ebc440b73c41eea450c593dbce11523a5',
 'HEAD_BYTES': 1051117,
 'HEAD_DIM': 512,
 'HEAD_LOCK_PATH': '',
 'HEAD_LOCK_SHA256': '60040d04c9b86a94db2b758c50d56b1c24c84971caf56bde3f14de37d2c7e752',
 'TRAIN_REPORT_PATH': '',
 'TRAIN_REPORT_SHA256': 'd8eab730c28499b3e54f04aa20a4530695a74565496bf5efa0fb57b056a623eb',
 'TRAIN_EXPERIMENT_ID': '0b1aea8b67054ec5719e93428394bac44a363ff14f55e674224b3bf19021f2a2',
 'TRAIN_RESULT_ZIP_SHA256': '52b9167686590a8c8b31fe6db90ebbe1d9e12dc92b444298e34a7a866a22be99',
 'BASELINE_REFERENCE_NPZ': '',
 'BASELINE_REFERENCE_SHA256': '429403c137f57f98ff93d7f3fcf121e32a5c3b88f0b60688c784a2028eaa15ba',
 'REFERENCE_IS_LOGITS': True,
 'REQUIRE_EXTERNAL_REFERENCE': True,
 'ALLOW_UNVERIFIED_EXPLORATORY': False,
 'EXPECTED_TEST_QUERIES': 1000,
 'EXPECTED_TEST_VIDEOS': 1000,
 'EXPECTED_BASELINE_R1': 41.0,
 'EXPECTED_R1_TOL_PP': 1e-08,
 'RUN_TRAIN': False,
 'SELECTION_POLICY': 'fixed_epochs',
 'PRECISION': 'fp32',
 'MAX_FRAMES': 12,
 'MAX_WORDS': 32,
 'ENCODER_BATCH': 4,
 'TEXT_BATCH': 1,
 'QUERY_CHUNK': 32,
 'VIDEO_CHUNK': 32,
 'TAU': 0.05,
 'ALPHA': 0.25,
 'SEED': 42,
 'BOOTSTRAP_SAMPLES': 2000,
 'ATOL': 1e-05,
 'RTOL': 1e-05,
 'ALLOW_CPU_DEBUG': False,
 'MIN_FREE_DISK_GIB': 2.0}
CONFIG["WORK_DIR"] = str(base / "public_evaluation_run")


In [ ]:
# Thay payload nhúng bằng mã công khai có SHA256; input dữ liệu vẫn được gắn riêng.
import importlib.util

PUBLIC_CODE = Path("/kaggle/working/temporal-hoi-video-text-matching/scripts/kaggle/msrvtt")
_spec = importlib.util.spec_from_file_location("prepare_public_sources", PUBLIC_CODE / "prepare_sources.py")
_prepare = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_prepare)
_prepare.restore_sources(PUBLIC_CODE, CODE_DIR)
for _module in ("data_pipeline", "retrieval_core", "workflow", "resources", "bpe_bootstrap", "b3_probe_runtime", "fulltrain_runtime", "b3_eval_runtime"):
    sys.modules.pop(_module, None)
VENDOR_DIR = CODE_DIR / "vendor"
_prepare.capture_stage(PUBLIC_CODE, CODE_DIR, Path(CONFIG["WORK_DIR"]), "evaluation", Path(CONFIG["INPUT_ROOT"]))
_work = Path(CONFIG["WORK_DIR"])


In [ ]:
# Cell 4 — Kiểm head, cache, CSV, reference trước khi chấm.
from b3_eval_runtime import prepare_evaluation

EVAL_STATE = prepare_evaluation(CONFIG)
print("Input/head/cache verification passed. No B3 scores computed in this cell.")


In [ ]:
# Cell 5 — B0 parity gate, rồi B1/B2/B3 và so sánh hai chiều.
from b3_eval_runtime import run_evaluation

EVAL_REPORT = run_evaluation(CONFIG, VENDOR_DIR)
print("Status:", EVAL_REPORT["status"])
print("Đọc b3_evaluation_report.json và các CSV trong Output để xem đầy đủ kết quả.")


In [ ]:
# Cell 6 — Chỉ xuất artifact của lượt hiện tại, đã đối chiếu hash.
from b3_eval_runtime import export_evaluation_results

_report_path = _work / "b3_evaluation_report.json"
if not _report_path.is_file():
    raise FileNotFoundError("Chưa có report; kiểm Cell5 trước khi xuất.")
_saved_report = json.loads(_report_path.read_text("utf-8"))
result_zip = export_evaluation_results(
    _work, base / "msrvtt_b3_test1k_results.zip"
)
print("Report status:", _saved_report["status"])
print("Tải ZIP này và gửi để phân tích:", result_zip)
if importlib.util.find_spec("IPython") is not None:
    from IPython.display import FileLink, display
    display(FileLink(str(result_zip)))
